# Гипотеза: Влияние частых командировок на текучесть персонала

## Данные
Таблица `V4_Employee_cleaned.csv` (предобработанная на этапе V4).

## Цель
Проверить гипотезу: сотрудники, часто выезжающие в командировки (`Frequent Traveller`), увольняются чаще, чем сотрудники, которые не ездят или ездят время от времени.

**Единица наблюдения:** один сотрудник (`employeeid`).

**Основное сравнение:** `Frequent Traveller` против объединённой группы `Some Travel` + `No Travel`.

**Критерии:** уровень значимости α = 0.05; порог практически значимого эффекта — 5 процентных пунктов.


## 1. Импорт библиотек и параметры

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display


ALPHA = 0.05
PRACTICAL_SIGNIFICANCE_PP = 5.0

TARGET = "attrition_flag"       # 0 = остался, 1 = уволился
PREDICTOR = "businesstravel"
EMPLOYEE_KEY = "employeeid"

FREQUENT = "Frequent Traveller"
SOME = "Some Travel"
NO_TRAVEL = "No Travel"


## 2. Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

path = '/content/drive/MyDrive/Проект команды 3/'
file_path = os.path.join(path, "V4_Employee_cleaned.csv")

df = pd.read_csv(file_path)

print(f"Загружено строк: {len(df)}")
print(f"Размер таблицы: {df.shape}")


Mounted at /content/drive
Загружено строк: 1470
Размер таблицы: (1470, 46)


## 3. Структура данных и проверка качества

In [ ]:
print("=" * 70)
print("1. СТРУКТУРА ДАННЫХ")
print("=" * 70)

print(f"Размер таблицы: {df.shape[0]} строк × {df.shape[1]} столбцов")
print(f"Ключ employeeid уникален: {df[EMPLOYEE_KEY].is_unique}")
print(f"Дубликаты employeeid: {df.duplicated(subset=[EMPLOYEE_KEY]).sum()}")

print("\nТипы ключевых полей:")
for col in [EMPLOYEE_KEY, PREDICTOR, TARGET, "department"]:
    if col in df.columns:
        print(f"{col}: {df[col].dtype}")

print("\nПропуски:")
for col in [PREDICTOR, TARGET, "department"]:
    if col in df.columns:
        n_missing = int(df[col].isna().sum())
        print(f"{col}: {n_missing} ({n_missing / len(df) * 100:.1f}%)")

print("\nЗначения businesstravel:")
display(df[PREDICTOR].value_counts(dropna=False).to_frame("Количество"))

print("\nЗначения attrition_flag:")
display(df[TARGET].value_counts(dropna=False).to_frame("Количество"))


1. СТРУКТУРА ДАННЫХ
Размер таблицы: 1470 строк × 46 столбцов
Ключ employeeid уникален: True
Дубликаты employeeid: 0

Типы ключевых полей:
employeeid: object
businesstravel: object
attrition_flag: int64
department: object

Пропуски:
businesstravel: 0 (0.0%)
attrition_flag: 0 (0.0%)
department: 0 (0.0%)

Значения businesstravel:


,Количество
businesstravel,
Some Travel,1043
Frequent Traveller,277
No Travel,150



Значения attrition_flag:


,Количество
attrition_flag,
0,1233
1,237


## 4. Подготовка групп для сравнения

In [ ]:

expected_categories = {FREQUENT, SOME, NO_TRAVEL}
observed_categories = set(df[PREDICTOR].dropna().unique())

print("Найденные категории:", observed_categories)

if not expected_categories.issubset(observed_categories):
    print("ВНИМАНИЕ: категории отличаются от ожидаемых. Проверьте написание значений.")

def travel_group(value):
    if value == FREQUENT:
        return "Частые командировки"
    elif value in [SOME, NO_TRAVEL]:
        return "Нет частых командировок"
    else:
        return np.nan

analysis_df = df.copy()
analysis_df["travel_group"] = analysis_df[PREDICTOR].apply(travel_group)

analysis_df = analysis_df.dropna(subset=["travel_group", TARGET])
analysis_df = analysis_df[analysis_df[TARGET].isin([0, 1])].copy()

print("\nКоличество сотрудников в группах:")
display(analysis_df["travel_group"].value_counts().to_frame("Количество"))


Найденные категории: {'Some Travel', 'Frequent Traveller', 'No Travel '}
ВНИМАНИЕ: категории отличаются от ожидаемых. Проверьте написание значений.

Количество сотрудников в группах:


,Количество
travel_group,
Нет частых командировок,1043
Частые командировки,277


## 5. Кросс-таблица и доли увольнений

In [ ]:
print("=" * 70)
print("2. КРОСС-ТАБЛИЦА: КОМАНДИРОВКИ × ТЕКУЧЕСТЬ")
print("=" * 70)

ct = pd.crosstab(analysis_df["travel_group"], analysis_df[TARGET])
ct = ct.reindex(
    ["Частые командировки", "Нет частых командировок"],
    fill_value=0
).reindex(columns=[0, 1], fill_value=0)

ct.columns = ["Остались (0)", "Уволились (1)"]
ct["Всего"] = ct.sum(axis=1)
ct["Текучесть, %"] = (
    ct["Уволились (1)"] / ct["Всего"] * 100
).round(2)

display(ct)

frequent = analysis_df.loc[
    analysis_df["travel_group"] == "Частые командировки", TARGET
].astype(int)

not_frequent = analysis_df.loc[
    analysis_df["travel_group"] == "Нет частых командировок", TARGET
].astype(int)

n_frequent = len(frequent)
n_not_frequent = len(not_frequent)

rate_frequent = frequent.mean() * 100 if n_frequent else np.nan
rate_not_frequent = not_frequent.mean() * 100 if n_not_frequent else np.nan

diff_pp = rate_frequent - rate_not_frequent

print(f"Текучесть при частых командировках: {rate_frequent:.2f}% (n={n_frequent})")
print(f"Текучесть без частых командировок: {rate_not_frequent:.2f}% (n={n_not_frequent})")
print(f"Разница (частые − остальные): {diff_pp:.2f} п.п.")


2. КРОСС-ТАБЛИЦА: КОМАНДИРОВКИ × ТЕКУЧЕСТЬ


,Остались (0),Уволились (1),Всего,"Текучесть, %"
travel_group,,,,
Частые командировки,208,69,277,24.91
Нет частых командировок,887,156,1043,14.96


Текучесть при частых командировках: 24.91% (n=277)
Текучесть без частых командировок: 14.96% (n=1043)
Разница (частые − остальные): 9.95 п.п.


## 6. Статистическая значимость: χ²-тест по трём категориям

In [ ]:
# Строим таблицу сопряжённости
ct_global = pd.crosstab(
    df["businesstravel"],
    df["attrition"]
)

# Убираем пустые строки и столбцы
ct_global = ct_global.loc[
    ct_global.sum(axis=1) > 0,
    ct_global.sum(axis=0) > 0
]

print("Таблица сопряжённости:")
display(ct_global)

# Проверяем, что таблица подходит для критерия хи-квадрат
if ct_global.shape[0] < 2 or ct_global.shape[1] < 2:
    print("Недостаточно категорий для проведения теста хи-квадрат.")
else:
    chi2, p_chi, dof, expected = stats.chi2_contingency(ct_global)

    print(f"χ² = {chi2:.4f}")
    print(f"p-value = {p_chi:.4f}")
    print(f"Степени свободы = {dof}")

    print("\nОжидаемые частоты:")
    display(pd.DataFrame(
        expected,
        index=ct_global.index,
        columns=ct_global.columns
    ))

    if p_chi < 0.05:
        print("Вывод: есть статистически значимая связь между командировками и увольнением.")
    else:
        print("Вывод: статистически значимой связи между командировками и увольнением не обнаружено.")



Таблица сопряжённости:


attrition,No,Yes
businesstravel,,
Frequent Traveller,208,69
No Travel,138,12
Some Travel,887,156


χ² = 24.1824
p-value = 0.0000
Степени свободы = 2

Ожидаемые частоты:


attrition,No,Yes
businesstravel,,
Frequent Traveller,232.340816,44.659184
No Travel,125.816327,24.183673
Some Travel,874.842857,168.157143


Вывод: есть статистически значимая связь между командировками и увольнением.


## 7. Парное сравнение: Z-тест двух долей

In [ ]:
print("=" * 70)
print("4. ПАРНОЕ СРАВНЕНИЕ: ЧАСТЫЕ КОМАНДИРОВКИ VS ОСТАЛЬНЫЕ")
print("=" * 70)

# H0: доли увольнений равны
# H1: доли увольнений различаются (двусторонняя проверка)

x1 = int(frequent.sum())
x2 = int(not_frequent.sum())

if n_frequent == 0 or n_not_frequent == 0:
    raise ValueError("Одна из групп пуста — невозможно провести сравнение.")

p_pool = (x1 + x2) / (n_frequent + n_not_frequent)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_frequent + 1/n_not_frequent))

if se_pool == 0:
    z_stat = 0.0
    p_z = 1.0
else:
    z_stat = (x1/n_frequent - x2/n_not_frequent) / se_pool
    p_z = 2 * stats.norm.sf(abs(z_stat))

print(f"Уволившиеся при частых командировках: {x1} из {n_frequent}")
print(f"Уволившиеся без частых командировок: {x2} из {n_not_frequent}")
print(f"Z-статистика = {z_stat:.4f}")
print(f"p-value = {p_z:.6f}")

if p_z < ALPHA:
    print("Результат статистически значим: доли увольнений различаются.")
else:
    print("Результат статистически незначим: оснований утверждать о различии долей нет.")


4. ПАРНОЕ СРАВНЕНИЕ: ЧАСТЫЕ КОМАНДИРОВКИ VS ОСТАЛЬНЫЕ
Уволившиеся при частых командировках: 69 из 277
Уволившиеся без частых командировок: 156 из 1043
Z-статистика = 3.9158
p-value = 0.000090
Результат статистически значим: доли увольнений различаются.


## 8. Практически значимый эффект

In [ ]:
print("=" * 70)
print("5. ПРАКТИЧЕСКАЯ ЗНАЧИМОСТЬ")
print("=" * 70)

print(f"Порог практически значимого эффекта: {PRACTICAL_SIGNIFICANCE_PP:.1f} п.п.")
print(f"Полученная разница: {diff_pp:.2f} п.п.")
pract_significant = abs(diff_pp) >= PRACTICAL_SIGNIFICANCE_PP
print(f"Практически значим: {'ДА' if pract_significant else 'НЕТ'}")


5. ПРАКТИЧЕСКАЯ ЗНАЧИМОСТЬ
Порог практически значимого эффекта: 5.0 п.п.
Полученная разница: 9.95 п.п.
Практически значим: ДА


## 9. Перепроверка по отделам

In [ ]:
print("=" * 70)
print("6. ПЕРЕПРОВЕРКА ПО СЕГМЕНТАМ (ОТДЕЛЫ)")
print("=" * 70)

if "department" in analysis_df.columns:
    strata_rows = []

    for dept, sub in analysis_df.groupby("department"):
        s_frequent = sub.loc[
            sub["travel_group"] == "Частые командировки", TARGET
        ].astype(int)

        s_not_frequent = sub.loc[
            sub["travel_group"] == "Нет частых командировок", TARGET
        ].astype(int)

        if len(s_frequent) > 0 and len(s_not_frequent) > 0:
            strata_rows.append({
                "department": dept,
                "n_frequent": len(s_frequent),
                "rate_frequent_%": round(s_frequent.mean() * 100, 1),
                "n_not_frequent": len(s_not_frequent),
                "rate_not_frequent_%": round(s_not_frequent.mean() * 100, 1),
                "diff_pp": round(
                    (s_frequent.mean() - s_not_frequent.mean()) * 100, 1
                )
            })

    strata_df = pd.DataFrame(strata_rows)

    if not strata_df.empty:
        display(strata_df.sort_values("diff_pp", ascending=False))
    else:
        print("Недостаточно данных для сравнения по отделам.")
else:
    print("Столбец department отсутствует — сегментация пропущена.")


6. ПЕРЕПРОВЕРКА ПО СЕГМЕНТАМ (ОТДЕЛЫ)


,department,n_frequent,rate_frequent_%,n_not_frequent,rate_not_frequent_%,diff_pp
0,Human Resources,11,36.4,46,17.4,19.0
1,Sales,84,33.3,315,19.0,14.3
2,Technology,182,20.3,682,12.9,7.4


## 10. Наблюдение / Интерпретация / Влияние / Ограничение

In [ ]:
print("=" * 70)
print("7. НАБЛЮДЕНИЕ / ИНТЕРПРЕТАЦИЯ / ВЛИЯНИЕ / ОГРАНИЧЕНИЕ")
print("=" * 70)

print(
    f"1) Наблюдение: текучесть при частых командировках — {rate_frequent:.1f}%, "
    f"без частых командировок — {rate_not_frequent:.1f}%. "
    f"Разница — {diff_pp:.1f} п.п."
)

print(
    "2) Интерпретация: результат может указывать на связь частоты командировок "
    "с увольнением, но не доказывает причинно-следственную зависимость."
)

if diff_pp > 0:
    direction = "выше"
elif diff_pp < 0:
    direction = "ниже"
else:
    direction = "равна"

print(
    f"3) Влияние на гипотезу: доля увольнений среди часто командирующихся "
    f"сотрудников {direction}, чем среди остальных."
)

print(
    "4) Ограничение уверенности: анализ основан на наблюдательных данных. "
    "Возможное влияние могут оказывать должность, отдел, стаж, зарплата и "
    "другие характеристики сотрудников. Нельзя утверждать, что командировки "
    "являются причиной увольнения."
)


7. НАБЛЮДЕНИЕ / ИНТЕРПРЕТАЦИЯ / ВЛИЯНИЕ / ОГРАНИЧЕНИЕ
1) Наблюдение: текучесть при частых командировках — 24.9%, без частых командировок — 15.0%. Разница — 10.0 п.п.
2) Интерпретация: результат может указывать на связь частоты командировок с увольнением, но не доказывает причинно-следственную зависимость.
3) Влияние на гипотезу: доля увольнений среди часто командирующихся сотрудников выше, чем среди остальных.
4) Ограничение уверенности: анализ основан на наблюдательных данных. Возможное влияние могут оказывать должность, отдел, стаж, зарплата и другие характеристики сотрудников. Нельзя утверждать, что командировки являются причиной увольнения.


## 11. Итоговый вердикт

In [ ]:
print("=" * 70)
print("8. ИТОГОВЫЙ ВЕРДИКТ")
print("=" * 70)

stat_significant = p_z < ALPHA

if diff_pp > 0 and stat_significant and pract_significant:
    verdict = "ПОДТВЕРЖДЕНА"
    recommendation = (
        "Стоит дополнительно изучить нагрузку сотрудников, часто выезжающих "
        "в командировки, и проверить возможные меры по снижению нагрузки."
    )
elif not stat_significant:
    verdict = "НЕ ПОДТВЕРЖДЕНА (нет статистически значимой связи)"
    recommendation = (
        "По этим данным нет достаточных оснований считать частые командировки "
        "значимым фактором текучести."
    )
elif not pract_significant:
    verdict = "НЕ ПОДТВЕРЖДЕНА (эффект ниже практического порога)"
    recommendation = (
        "Разница ниже заранее установленного порога практической значимости; "
        "не стоит считать этот фактор приоритетным без дополнительных данных."
    )
else:
    verdict = "ТРЕБУЕТ УТОЧНЕНИЯ"
    recommendation = (
        "Получены неоднозначные результаты. Необходимо проверить группы и "
        "возможные сопутствующие факторы."
    )

print(f"Вердикт: {verdict}")
print(f"Рекомендация заказчику: {recommendation}")
print(
    f"Критерии: разница в ожидаемом направлении, p-value < {ALPHA} "
    f"и абсолютная разница не менее {PRACTICAL_SIGNIFICANCE_PP:.1f} п.п."
)


8. ИТОГОВЫЙ ВЕРДИКТ
Вердикт: ПОДТВЕРЖДЕНА
Рекомендация заказчику: Стоит дополнительно изучить нагрузку сотрудников, часто выезжающих в командировки, и проверить возможные меры по снижению нагрузки.
Критерии: разница в ожидаемом направлении, p-value < 0.05 и абсолютная разница не менее 5.0 п.п.


## 12. Резюме

**1. Сформулированная гипотеза**

Сотрудники, часто выезжающие в командировки (`Frequent Traveller`), увольняются чаще, чем сотрудники, которые не ездят или ездят время от времени (`Some Travel` и `No Travel`).

**2. Обоснование гипотезы**

Частые командировки потенциально могут увеличивать рабочую нагрузку, затруднять поддержание баланса между работой и личной жизнью и повышать утомляемость сотрудников.

**3. План проверки**

1. Проверить структуру данных, уникальность `employeeid` и пропуски.
2. Проверить категории `businesstravel` и значения `attrition_flag`.
3. Объединить `Some Travel` и `No Travel` в группу без частых командировок.
4. Построить кросс-таблицу и рассчитать долю увольнений.
5. Провести глобальный χ²-тест для трёх исходных категорий.
6. Провести парный Z-тест для двух групп.
7. Сопоставить разницу с порогом практической значимости 5 п.п.
8. Проверить результаты по отделам.
9. Сформулировать вердикт.

**4. Принципиальная опровержимость**

Гипотеза не подтверждается, если доля увольнений среди часто командирующихся сотрудников не выше, либо различие статистически незначимо, либо эффект меньше установленного практического порога.

**5. Ограничения**

Данные позволяют выявить статистическую связь, но не доказывают причинность. Различия могут объясняться должностями, подразделениями, стажем, уровнем оплаты и другими факторами. Также результаты по отделам следует интерпретировать осторожно из-за небольших размеров отдельных групп.
